# Practice 1 · Data Pipeline in Code

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

Before a network sees a single image, three things have to be in place: the **transforms**
(crop / flip / to tensor / normalise), the **dataset** that applies them lazily, and the
**loader** that groups samples into batches. This practice builds that pipeline and then
measures what comes out of it.

Normalise with the CIFAR10 statistics *MEAN = (0.4914, 0.4822, 0.4465)* and
*STD = (0.2470, 0.2435, 0.2613)*.

### II. Task Requirements

1. **Q1** — write *build_transforms()*: the training pipeline and the test pipeline.
2. **Q2** — write the dataset class *CifarSubset(base, n, transform)* and the helper
   *channel_stats(loader)*.
3. Run every cell and keep the printed numbers: they are what the report asks for.

Steps: open the notebook in Haima, write the 🟢 parts, run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *train_tf*, *test_tf* | the two pipelines | *transforms.Compose* |
| one batch | 64 training images | *(64, 3, 32, 32)* |
| mean / std before *Normalize* | per channel | about 0.42–0.48 / 0.24–0.26 |
| mean / std after *Normalize* | per channel | close to 0 / close to 1 |
| *len(CifarSubset(train_raw, 10))* | dataset length | 10 |
| one sample | *(image, label)* | *(3, 32, 32)* and an int |

### IV. Who does what

- **Q1**: required for undergraduate students (postgraduates do it as well).
- **Q2**: required for postgraduate students; optional for undergraduates.
  An undergraduate who also finishes Q2 **gets bonus marks**; leaving it unfinished
  **costs nothing**.


## Question 1 · Build the transforms

Write *build_transforms()* so that it returns two *transforms.Compose* objects:

- *train_tf*: *RandomCrop(32, padding=4)* → *RandomHorizontalFlip()* → *ToTensor()* → *Normalize(MEAN, STD)*
- *test_tf*: *ToTensor()* → *Normalize(MEAN, STD)*

The test transform must be deterministic, the training transform must be random.
The cell below then prints one batch and compares the statistics **before and after**
*Normalize*.


In [1]:
# 🟢 EDIT: write the two pipelines inside build_transforms() and run the cell.
# 🟡 READ: everything outside the 🟢 block is given - just run it.
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from PIL import Image

torch.manual_seed(0)

# 🟡 READ: CIFAR10 ships with the assignment; ./data holds cifar-10-batches-py.
#   Nothing is downloaded here - the notebook stops with a message if the folder is missing.
DATA_ROOT = "data"
if not os.path.exists(os.path.join(DATA_ROOT, "cifar-10-batches-py")):
    raise FileNotFoundError(
        "CIFAR-10 not found under ./data - please contact the teaching assistant "
        "and copy the folder cifar-10-batches-py next to this notebook."
    )

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2613)

train_raw = CIFAR10(DATA_ROOT, train=True, download=False, transform=None)
print("train =", len(train_raw))


def build_transforms():
    """Return (train_tf, test_tf) for CIFAR10."""
    # 🟢 CODE HERE - 3 statements:
    #   1) train_tf = transforms.Compose([...])   the training pipeline, 4 steps in this order:
    #        transforms.RandomCrop(32, padding=4) -> transforms.RandomHorizontalFlip()
    #        -> transforms.ToTensor() -> transforms.Normalize(MEAN, STD)
    #   2) test_tf  = transforms.Compose([...])   the test pipeline, only 2 steps:
    #        transforms.ToTensor() -> transforms.Normalize(MEAN, STD)
    #   3) return both objects
    #   Compose takes a list and applies the steps in order.
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, test_tf


# ⚪ RUN: one batch through the training pipeline
train_tf, test_tf = build_transforms()
train_ds = CIFAR10(DATA_ROOT, train=True, download=False, transform=train_tf)
xb, yb = next(iter(DataLoader(train_ds, batch_size=64, shuffle=True)))
print("one batch:", tuple(xb.shape), tuple(yb.shape))

# ⚪ RUN: the same 64 images, before and after Normalize
#   shuffle=False on both sides, so the two batches hold exactly the same pictures.
raw_ds = CIFAR10(DATA_ROOT, train=True, download=False, transform=transforms.ToTensor())
nrm_ds = CIFAR10(DATA_ROOT, train=True, download=False, transform=test_tf)
x_raw = next(iter(DataLoader(raw_ds, batch_size=64, shuffle=False)))[0]
x_nrm = next(iter(DataLoader(nrm_ds, batch_size=64, shuffle=False)))[0]

raw_mean = x_raw.mean(dim=(0, 2, 3))
nrm_mean = x_nrm.mean(dim=(0, 2, 3))
print("before Normalize : mean =", [round(v, 4) for v in raw_mean.tolist()],
      "| std =", [round(v, 4) for v in x_raw.std(dim=(0, 2, 3)).tolist()])
print("after  Normalize : mean =", [round(v, 4) for v in nrm_mean.tolist()],
      "| std =", [round(v, 4) for v in x_nrm.std(dim=(0, 2, 3)).tolist()])
print("check (raw - MEAN) / STD =",
      [round(float((r - m) / s), 4) for r, m, s in zip(raw_mean, MEAN, STD)])


train = 50000


one batch: (64, 3, 32, 32) (64,)


before Normalize : mean = [0.4816, 0.4646, 0.4168] | std = [0.248, 0.2408, 0.2481]
after  Normalize : mean = [-0.0399, -0.0722, -0.1137] | std = [1.0039, 0.9891, 0.9495]
check (raw - MEAN) / STD = [-0.0399, -0.0722, -0.1137]


## Question 2 · A dataset of your own

*CIFAR10* is a *Dataset* already; the point here is to see what a *Dataset* owes the loader.
Write a subclass *CifarSubset(base, n, transform=None)* that exposes the **first n** samples of
*base* (a *CIFAR10* object with *transform=None*), and a helper *channel_stats(loader)* that
returns the per-channel *mean* and *std* of one pass over a *DataLoader* — both as tensors of
shape *(3,)*.

A *Dataset* must give a length, an *(image, label)* pair per index, and apply its transform.
*__init__* is already written for you; you fill in *__len__* and *__getitem__*.
*channel_stats* must work for any loader whose batches are *(B, 3, H, W)* float tensors.


In [2]:
# 🟢 EDIT: fill in the two remaining methods of CifarSubset and the body of channel_stats().
# 🟡 READ: train_raw, test_tf, torch, Dataset and DataLoader come from the Q1 cell above.
class CifarSubset(Dataset):
    """First n samples of a CIFAR10 split."""

    # 🟢 CODE HERE - two methods, about 5 lines in total:
    #   __len__(self)          : return the number of samples                        (1 line)
    #   __getitem__(self, i)   : img, label = self.base[i]
    #                            apply self.transform when it is not None
    #                            return (img, label)                                (4 lines)
    #   base[i] returns a PIL image, so the transform you built in Q1 can be used as it is.

    # 🟡 READ: __init__ is given - it just stores the three arguments on self
    def __init__(self, base, n, transform=None):
        self.base = base
        self.n = n
        self.transform = transform

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        img, label = self.base[i]
        if self.transform is not None:
            img = self.transform(img)
        return img, label


def channel_stats(loader):
    """Per-channel mean and std over one pass, both with shape (3,)."""
    # 🟢 CODE HERE - 4 parts:
    #   1) three accumulators: number of pixels, sum of pixel values, sum of their squares
    #   2) for x, _ in loader:   x = x.float();  add the batch into the three accumulators
    #        (x.sum(dim=(0, 2, 3)) gives one number per channel;
    #         x.shape[0] * x.shape[2] * x.shape[3] is the number of pixels in the batch)
    #   3) mean = sum / number of pixels
    #   4) std  = sqrt(sum of squares / number of pixels - mean ** 2)
    #        torch.clamp_min(..., 0) keeps a tiny negative value from breaking the sqrt
    #   return (mean, std), both of shape (3,)
    num_pixels = 0
    channel_sum = 0
    channel_sq_sum = 0
    for x, _ in loader:
        x = x.float()
        num_pixels += x.shape[0] * x.shape[2] * x.shape[3]
        channel_sum += x.sum(dim=(0, 2, 3))
        channel_sq_sum += (x ** 2).sum(dim=(0, 2, 3))
    mean = channel_sum / num_pixels
    std = torch.sqrt(torch.clamp_min(channel_sq_sum / num_pixels - mean ** 2, 0))
    return mean, std


# ⚪ RUN: 512 training images through your dataset
sub = CifarSubset(train_raw, 512, test_tf)
dl = DataLoader(sub, batch_size=64)
m, s = channel_stats(dl)
print("mean =", [round(v, 4) for v in m.tolist()])
print("std  =", [round(v, 4) for v in s.tolist()])


mean = [-0.0093, -0.0135, -0.0122]
std  = [1.0031, 0.9975, 0.9885]
